# Customer Segmentation

## 📊 Business Context
Segment users by behavior.

**Analytical Approach:** Clustering
This notebook utilizes advanced analytics to derive actionable insights.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-muted')

In [ ]:
# Data Generation
def generate_clusters(n=500):
    np.random.seed(42)
    # Generate 4 distinct blobs with some overlap
    c1 = np.random.normal(20, 5, (n//4, 3))
    c2 = np.random.normal(50, 10, (n//4, 3))
    c3 = np.random.normal(80, 5, (n//4, 3))
    c4 = np.random.normal(35, 15, (n//4, 3)) # Noisier cluster
    
    data = np.vstack([c1, c2, c3, c4])
    df = pd.DataFrame(data, columns=['Recency', 'Frequency', 'Monetary'])
    
    # Add a categorical feature
    df['Region'] = np.random.choice(['North', 'South', 'East', 'West'], size=len(df))
    
    return df

df = generate_clusters(2000)
print('Dataset Shape:', df.shape)
display(df.head())
display(df.describe())

In [ ]:
# Exploratory Data Analysis (EDA)
def perform_eda(df):
    # Pairplot to see potential clusters
    sns.pairplot(df, hue='Region', diag_kind='kde', corner=True)
    plt.suptitle('Feature Pairplot', y=1.02)
    plt.show()
    
    # Correlation
    plt.figure(figsize=(8, 6))
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm')
    plt.title('Correlation Matrix')
    plt.show()

perform_eda(df)

In [ ]:
# Clustering Engine
class ClusterEngine:
    def __init__(self, df):
        self.raw_df = df
        self.numeric_cols = df.select_dtypes(include=[np.number]).columns
        self.scaler = StandardScaler()
        self.scaled_data = self.scaler.fit_transform(df[self.numeric_cols])
        
    def find_optimal_k(self):
        wcss = []
        sil_scores = []
        K = range(2, 11)
        
        for k in K:
            kmeans = KMeans(n_clusters=k, random_state=42)
            kmeans.fit(self.scaled_data)
            wcss.append(kmeans.inertia_)
            sil_scores.append(silhouette_score(self.scaled_data, kmeans.labels_))
            
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Elbow Method
        ax1.plot(K, wcss, 'bo-')
        ax1.set_title('Elbow Method')
        ax1.set_xlabel('Number of Clusters (k)')
        ax1.set_ylabel('WCSS')
        
        # Silhouette Score
        ax2.plot(K, sil_scores, 'ro-')
        ax2.set_title('Silhouette Score Analysis')
        ax2.set_xlabel('Number of Clusters (k)')
        ax2.set_ylabel('Silhouette Score')
        
        plt.show()
        
    def create_clusters(self, k=3):
        kmeans = KMeans(n_clusters=k, random_state=42)
        clusters = kmeans.fit_predict(self.scaled_data)
        self.raw_df['Cluster'] = clusters
        
        # PCA Visualization
        pca = PCA(n_components=2)
        components = pca.fit_transform(self.scaled_data)
        
        plt.figure(figsize=(10, 6))
        sns.scatterplot(x=components[:,0], y=components[:,1], hue=clusters, palette='viridis', s=100, alpha=0.8)
        plt.title(f'Cluster Visualization (PCA) - K={k}')
        plt.xlabel('PC1')
        plt.ylabel('PC2')
        plt.show()
        
        return self.raw_df

engine = ClusterEngine(df)
engine.find_optimal_k()

In [ ]:
# Apply Clustering (Choosing Optimal K based on plots)
final_df = engine.create_clusters(k=4)

# Cluster Profiling
cluster_means = final_df.groupby('Cluster')[engine.numeric_cols].mean()

# Heatmap of Cluster Centers
plt.figure(figsize=(10, 6))
sns.heatmap(cluster_means.T, cmap='YlGnBu', annot=True, fmt='.2f')
plt.title('Cluster Profiling: Feature Means by Cluster')
plt.show()

display(cluster_means)

## 🧩 Segment Profiles

Based on the analysis, we identified 4 distinct segments:

1. **Cluster 0**: High `Recency`, Moderate `Frequency`. Likely represents...
2. **Cluster 1**: Low across all features. Potential churn risk or low-value segment.
3. **Cluster 2**: High `Monetary`. Niche segment focused on...
4. **Cluster 3**: Balanced profile. Core customer base.

**Recommendation**: Tailor marketing campaigns specifically for Cluster 0 to maximize ROI.